# Глубинное обучение на табличных данных

Начнем конечно с импортов различных, которые понадобятся далее. Также, многое отсюда было реализовано с помощью функций копипаста с семинаров, что мы проходили, помимо этого, семинары дали вдохновление на именно такой пайплайн действий) Стоило упомянуть

In [22]:
import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader #чтобы подавать данные батчами

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score #для подсчета качества

import os
import wandb

Объявим класс конфига

In [23]:
class CFG:
  project = "kolesa-cars"
  entity = "armntvs-d3v-student"
  num_epochs = 15
  train_batch_size = 64
  test_batch_size = 256
  lr = 0.001
  seed = 42
  wandb = True

In [24]:
# конфиг в словарь
def class2dict(f):
  return dict((name, getattr(f, name)) for name in dir(f) if not name.startswith('__'))

In [25]:
# вход в W&B (ключ спросит один раз)
wandb.login()

True

Зафиксируем сиды

In [26]:
def seed_everything(seed): # чтобы запуски были стабильными и воспроизводимымми
    random.seed(seed) # фиксируем генератор случайных чисел
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU

In [27]:
# https://stackoverflow.com/questions/63423463/using-pytorch-cuda-on-macbook-pro
# т.к. на macbook на их процессорах apple silicon нет cuda (только для карт nvidia), то используем альтеранативу, но если есть cuda - то используем ее
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

device

device(type='mps')

Загрузим все предобработанные данные 

In [28]:
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv')
y_test = pd.read_csv('../data/y_test.csv')

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4236, 570), (1060, 570), (4236, 1), (1060, 1))

Когда переводил в тензоры выдавало ошибку, потому что нужны флоат, а там видимо от ohe остались bool, переведем это все в флоат и пойдем дальше

In [29]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)
y_train = y_train.astype(float)
y_test = y_test.astype(float)

Переведем все в тензоры

In [30]:
X_train_tensor =torch.FloatTensor(X_train.values)
X_test_tensor =torch.FloatTensor(X_test.values)
y_train_tensor =torch.FloatTensor(y_train.values)
y_test_tensor =torch.FloatTensor(y_test.values)

X_train_tensor.shape, y_train_tensor.shape

(torch.Size([4236, 570]), torch.Size([4236, 1]))

Теперь соединим это обратно в датасеты (train/test), чтобы передавать в модели именно все признаки с таргетом, после чего создадим dataloader чтобы подавать в модель данные батчами/частями

In [31]:
train_ds = TensorDataset(X_train_tensor,y_train_tensor)
test_ds = TensorDataset(X_test_tensor,y_test_tensor)

train_ds

In [32]:
train_loader = DataLoader(train_ds, batch_size= CFG.train_batch_size, shuffle=True) #shuffle тут чтобы между эпохами перемешивались строки и модель не привыкала к порядку
test_loader = DataLoader(test_ds, batch_size= CFG.test_batch_size)


In [33]:
input_size = X_train.shape[1]
input_size 

570

Сейчас построим достаточно базовую модель (вдохновление от 15 семинара). Архитектура простая - 3 полносвязных слоя и все. Далее будем строить несколько более сложных архитектур. Сначала простую чтобы понимать, помогают ли нам новые архитектуры улучшить так скажем 'базовое' качество. 

In [34]:
class Simple(nn.Module): # наследуемся от класса nn.Module
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 128) # первый скрытый слой - 128 нейронов
        self.fc2 = nn.Linear(128, 64) # второй скрытый - 64
        self.fc3 = nn.Linear(64, 1) # третий  выходной- 1 прогноз
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x) 
        return x

Теперь построим более сложную (чуть) архитектуру. Попробуем добавить просто больше скрытых слоев и добавим еще больше нейронов 

Но, при этом, есильно выше риск переобучения, поэтому будем все это сравнивать на качестве теста

In [35]:
class Deep(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 512) #теперь начинаем уменьшение размерности с 512 
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

И, добавим третью модель, тут мы добавим еще дропаут и батчнорм. Первый будет бороться с переобучением за счет отключения определенного количества нейронов во время обучения. Батчнорм будет просто стабилизировать значения внутри срктытых слоев

In [36]:
class Regularized(nn.Module):
    def __init__(self, input_size):
        super().__init__()


        self.net = nn.Sequential(nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.05), #1 скрытый 
                                 nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),nn.Dropout(0.05),  #2 скрытый
                                 nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))    #3 скрытый и 4 выходной
        #sequential тут потому что во первых мы так работали на семинарах, а во-вторых потому что слои и операции в них просто идут по порядку
        #без сложностей , и следующая функция красивая в 2 строки получается )))

    def forward(self, x):
        return self.net(x)

Мы решили добавить еще одну полнсвязную сеть с residual connection. Его преимущество в том, что он сохраняет вход блока и добавляет к выходу. То есь, это хорошо потому что мы по идее не теряем юзфул информацию когда проходим несколько слоев. В нашем случае, это полезно потому что после ohe у нас стало много признаков, и там есть важные признаки ,которые residual mlp поможет учитывать в каких то сложных комбинациях

Сайты, в том числе откуда и вдохновение на код:
- https://discuss.pytorch.org/t/how-to-use-residual-learning-applied-to-fully-connected-networks/98708/5
- https://stackoverflow.com/questions/60817390/implementing-a-simple-resnet-block-with-pytorch

In [37]:
class ResBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        
    def forward(self, x):
        residual = x  # сохраняем вход блока
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = x + residual  # прибавляем старый x к новому x
        x = torch.relu(x)
        
        return x

In [38]:
class ResMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 256)
        self.block1 = ResBlock(256)
        self.block2 = ResBlock(256)
        self.fc2 = nn.Linear(256, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.fc2(x)
        return x

Далее, переходим к следующей части , пока просто зададим функцию потерь и потом напишем фукнцию обучения

In [39]:
criterion = nn.MSELoss()

In [40]:
from tqdm import tqdm

def train(model, device, train_loader, optimizer, criterion, epoch, WANDB): 
    model.train()
    train_loss = 0
    
    for data, target in tqdm(train_loader):
        data = data.to(device)
        target = target.to(device)
        
        optimizer.zero_grad() #обнуяем градиенты
        
        output = model(data) #прямой проход
        loss = criterion(output, target) #считаем ошибку
        
        loss.backward() #обратынй проход
        optimizer.step() #шаг оптимизатором
        train_loss += loss.item()
    
    train_loss = train_loss / len(train_loader)
    
    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(train_loss)) #как в семе, но чуть поправленная, потому что там для картинок

    if WANDB:
        wandb.log({'epoch': epoch,
                   'train_loss': train_loss})
        
    return train_loss

Теперь функция тестирования, оч похожа на сем, но там картинки

In [41]:
def test(model, device, test_loader, criterion, epoch, WANDB):
    model.eval()  
    
    test_loss = 0
    preds = []
    true = []
    
    # показываем, что обучения нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            test_loss += criterion(output, target).item()
            
            preds.append(output.cpu())
            true.append(target.cpu())
    
    test_loss = test_loss / len(test_loader)
    
    preds = torch.cat(preds).numpy().ravel() #тут мы клеим прогнозы в одномерный массив нампай 
    true = torch.cat(true).numpy().ravel()
    
    preds = np.expm1(preds) #возвращаем логарифмированные числа в первоначальное 
    true = np.expm1(true)
    
    mae = mean_absolute_error(true, preds)
    rmse = np.sqrt(mean_squared_error(true, preds))
    r2 = r2_score(true, preds)
    
    tqdm.write('Test set: Average loss: {:.4f}, MAE: {:.2f}, RMSE: {:.2f}, R2: {:.4f}'.format(
       test_loss, mae, rmse, r2))
    
    if WANDB:
        wandb.log({'epoch': epoch,
                   'test_loss': test_loss,
                   'test_mae': mae,
                   'test_rmse': rmse,
                   'test_r2': r2})
    
    return test_loss, mae, rmse, r2

Теперь нужна функция запуска наших экспериментов, и после этого можно будет лицезреть качество выстроенных нами моделей))

In [42]:
# основная функция для экспериментов
def main(model, model_name):

    if CFG.wandb:
        wandb.init(project=CFG.project, entity=CFG.entity, name=model_name, reinit=True, config=class2dict(CFG)) #reinit чтобы каждый экспер записывался как новый
        # параметры архитектуры https://docs.wandb.ai/guides/track/config
        wandb.config.update({'model': str(model)})
    seed_everything(CFG.seed)  # фиксируем сиды
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr)
    
    train_losses = []
    test_losses = []
    best_mae =10**20
    
    if CFG.wandb:
        wandb.watch(model, log='all') # логируем все (метрики, лоссы, градиенты)


    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train_loss = train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        test_loss, mae, rmse, r2 = test(model, device, test_loader, criterion, epoch, CFG.wandb)
        train_losses.append(train_loss)
        test_losses.append(test_loss)

        if mae < best_mae:
            best_mae = mae
            best_test_loss = test_loss
            best_rmse = rmse
            best_r2 = r2
    
    print('Training is ended!')
    
    
    if CFG.wandb:
        # сохраняем модель артефактом https://docs.wandb.ai/guides/artifacts
        torch.save(model.state_dict(), f'../data/models/{model_name}.pt')
        artifact = wandb.Artifact(model_name, type='model')
        artifact.add_file(f'../data/models/{model_name}.pt')
        wandb.log_artifact(artifact)
        wandb.finish()

    return model, train_losses, test_losses, best_test_loss, best_mae, best_rmse, best_r2

In [22]:
# данные в W&B для воспроизводимости https://docs.wandb.ai/guides/artifacts
# выполнить один раз
if CFG.wandb:
    wandb.init(project=CFG.project, entity=CFG.entity, name='dataset-tabular', reinit=True)
    art = wandb.Artifact('kolesa-tabular', type='dataset')
    for f in ['X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv']:
        art.add_file(f'../data/{f}')
    wandb.log_artifact(art)
    wandb.finish()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x1307c2a50>> (for post_run_cell), with arguments args (<ExecutionResult object at 134691080, execution_count=22 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 134766fd0, raw_cell="# данные в W&B для воспроизводимости https://docs..." transformed_cell="# данные в W&B для воспроизводимости https://docs..." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/ksa/smadimo/dev_gp5/gp_5_DL/notebooks/mlp_tab.ipynb#X50sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [43]:
seed_everything(CFG.seed)
simple_model, simple_train_losses, simple_test_losses, simple_test_loss, simple_mae, simple_rmse, simple_r2 = main(Simple(input_size),'Simple')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



Epoch: 1


100%|██████████| 67/67 [00:01<00:00, 48.77it/s]



Train set: Average loss: 154.6181
Test set: Average loss: 6.3434, MAE: 175640176.00, RMSE: 442042511.01, R2: -1896.2131

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 546.32it/s]


...


Train set: Average loss: 0.0755
Test set: Average loss: 0.2579, MAE: 2695520.75, RMSE: 5436532.98, R2: 0.7130

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 520.84it/s]



Train set: Average loss: 0.0675
Test set: Average loss: 0.2489, MAE: 2573330.25, RMSE: 5338706.86, R2: 0.7233
Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇██
test_loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁
test_mae,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_r2,▁██████████████
test_rmse,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,15
test_loss,0.24889
test_mae,2573330.25
test_r2,0.72327
test_rmse,5338706.86116


In [44]:
seed_everything(CFG.seed)
deep_model,deep_train_losses, deep_test_losses, deep_test_loss, deep_mae,deep_rmse, deep_r2 = main(Deep(input_size), 'Deep')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 107.96it/s]



Train set: Average loss: 69.2995
Test set: Average loss: 2.1083, MAE: 32222498.00, RMSE: 74257584.05, R2: -52.5389

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 414.16it/s]



Train set: Average loss: 0.9805
Test set: Average loss: 0.7312, MAE: 7087168.50, RMSE: 17028573.08, R2: -1.8154

Epoch: 3


...


Train set: Average loss: 0.0501
Test set: Average loss: 0.2127, MAE: 2456760.75, RMSE: 4940377.12, R2: 0.7630

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 342.59it/s]



Train set: Average loss: 0.0457
Test set: Average loss: 0.2174, MAE: 2350698.25, RMSE: 4476950.95, R2: 0.8054
Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇██
test_loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁
test_mae,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁
test_r2,▁██████████████
test_rmse,█▂▂▂▂▁▁▁▁▁▁▁▁▁▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,15
test_loss,0.21736
test_mae,2350698.25
test_r2,0.8054
test_rmse,4476950.95328


In [45]:
seed_everything(CFG.seed)
regularized_model,regularized_train_losses, regularized_test_losses, regularized_test_loss, regularized_mae,regularized_rmse, regularized_r2 = main(
    Regularized(input_size),'Regularized')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 85.08it/s] 



Train set: Average loss: 113.1711
Test set: Average loss: 4.6014, MAE: 9288961.00, RMSE: 13108109.41, R2: -0.6683

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 325.46it/s]



Train set: Average loss: 1.4206
Test set: Average loss: 0.8367, MAE: 6228206.00, RMSE: 9693206.72, R2: 0.0877

Epoch: 3


...


Train set: Average loss: 0.5583
Test set: Average loss: 0.2971, MAE: 4534983.00, RMSE: 7478348.49, R2: 0.4570

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 343.35it/s]



Train set: Average loss: 0.4687
Test set: Average loss: 0.4182, MAE: 5057628.00, RMSE: 7938428.16, R2: 0.3881
Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇██
test_loss,█▂▁▁▂▁▁▁▁▂▁▁▁▁▁
test_mae,█▄▃▂▃▂▃▂▂▄▁▂▂▂▃
test_r2,▁▅▇▆▆▆▆█▇▅██▇▇▇
test_rmse,█▅▃▄▃▄▄▂▂▅▁▂▂▂▃
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,15
test_loss,0.41817
test_mae,5057628.0
test_r2,0.38813
test_rmse,7938428.16014


In [46]:
seed_everything(CFG.seed)
res_model, res_train_losses, res_test_losses, res_test_loss, res_mae, res_rmse, res_r2 = main(ResMLP(input_size), 'ResMLP')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 133.37it/s]



Train set: Average loss: 41.0732
Test set: Average loss: 0.8385, MAE: 9280553.00, RMSE: 26109617.14, R2: -5.6189

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 356.24it/s]



Train set: Average loss: 0.5792
Test set: Average loss: 0.4251, MAE: 5570532.00, RMSE: 13822507.15, R2: -0.8551

Epoch: 3


...


Train set: Average loss: 0.0401
Test set: Average loss: 0.1541, MAE: 2607253.00, RMSE: 4554967.71, R2: 0.7986

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 289.95it/s]



Train set: Average loss: 0.0423
Test set: Average loss: 0.1499, MAE: 2369907.50, RMSE: 4301483.10, R2: 0.8204
Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇██
test_loss,█▄▃▂▂▁▁▁▁▁▁▁▁▁▁
test_mae,█▄▃▂▂▁▁▁▁▁▁▁▂▁▁
test_r2,▁▆█████████████
test_rmse,█▄▂▂▂▁▁▁▁▁▁▁▁▁▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,15
test_loss,0.14993
test_mae,2369907.5
test_r2,0.82035
test_rmse,4301483.10004


In [47]:
res = pd.DataFrame([{'model': 'Simple', 'test_loss': simple_test_loss, 'MAE': simple_mae, 'RMSE': simple_rmse, 'R2': simple_r2},
                        {'model': 'Deep', 'test_loss': deep_test_loss, 'MAE': deep_mae, 'RMSE': deep_rmse, 'R2': deep_r2},
                        {'model': 'Regularized', 'test_loss': regularized_test_loss, 'MAE': regularized_mae, 'RMSE': regularized_rmse, 'R2': regularized_r2},
                        {'model': 'ResMLP', 'test_loss': res_test_loss, 'MAE': res_mae, 'RMSE': res_rmse, 'R2': res_r2}])

res

,model,test_loss,MAE,RMSE,R2
0,Simple,0.248894,2573330.25,5.338707e+06,0.723267
1,Deep,0.217364,2350698.25,4.476951e+06,0.805396
2,Regularized,0.285035,3672826.75,5.971287e+06,0.653803
3,ResMLP,0.146879,2333615.00,4.335105e+06,0.817532


In [48]:
res.sort_values('MAE')

,model,test_loss,MAE,RMSE,R2
3,ResMLP,0.146879,2333615.00,4.335105e+06,0.817532
1,Deep,0.217364,2350698.25,4.476951e+06,0.805396
0,Simple,0.248894,2573330.25,5.338707e+06,0.723267
2,Regularized,0.285035,3672826.75,5.971287e+06,0.653803


### Сравнение моделей и выводы

По ходу работы мы обучили 4 модели, от простой базовой полносвязной сети до полносвязной с residual блоками. Основная метрика - считаем MAE, средняя ошибка прогноза цены в тенге. 

По результатам: лучшей стала модель ResMLP с показателями MAE - примерно 2,29 млн тенге, R2 - примерно 0,82, то есть модель лучше остальных отличает цены и в среднем меньше ошибается. Базовая модель(Simple) и модель с бОльшим количеством скрытых слоев(Deep) также дают хороший результат. Худший результат показала модель Regularized, возможно dropout для нашего датасета мешала модели запоминать какие-то важные зависимости в данных или еще что-то, но любые манипуляции с весами не помогали. 